# T-337 — Extractor distillation (gpt-oss → Qwen2.5-3B-Instruct)

**ResearchGPT**, local-model arm. Trains a **student extractor** small enough to
run on a 5 GB card, using the hosted `gpt-oss` extractions this project has
already paid for as labels.

| | |
|---|---|
| Base | `Qwen/Qwen2.5-3B-Instruct` (3.09B params) |
| Method | QLoRA — 4-bit NF4 base, LoRA r=16 on all seven projections |
| Teacher | `openai/gpt-oss-120b` (114 calls) + `openai/gpt-oss-20b` (60) |
| Labels | `data/llm_cache/*.json` — **162 cached extractor calls, 134 with a complete field set** |
| Hardware | Kaggle T4 16 GB (free). **Not trainable on the 5 GB card** — that machine is for inference |
| Deploy | GGUF `q4_K_M` → Ollama → `agents/extract.py` |

---

## ⚠️ Read this before running

**`TRAINING_SPEC.md §4` says the Extractor "cannot be fine-tuned at all in this
project: it needs 4-field labels that do not exist."** That statement was true
when written on 2026-08-25, and it referred to *human* labels — the 10-paper
annotation task cut in `PROJECT_PLAN.md §5.4`.

It is no longer the only option, because a second source of labels appeared as
a by-product of running the pipeline: **`data/llm_cache/` now holds 162
extractor responses from the teacher models**, produced at $0 under the free
tier and cached to disk. That is a distillation set, not an annotation set, and
the distinction matters for what may be claimed:

* A distilled student can be reported as **"reproduces the teacher's extraction
  behaviour at 3B"**.
* It can **never** be reported as more accurate than the teacher. Its ceiling
  *is* the teacher, including the teacher's errors — the measured 4.6–4.9 of 8
  field coverage is inherited, not fixed, by this notebook.

Amend `TRAINING_SPEC.md §4` before citing this in the report, or the two
documents contradict each other.

### What this buys the project

One thing, concretely: **an offline arm**. Groq's 200,000 tokens/day/model caps
the pipeline at roughly 2.5–3 runs a day (`eval/metrics_evaluated.md §4`). A
local student has no daily budget, so extraction stops being the rate limiter.
It also gives T-806 a real future-work paragraph instead of a hypothetical one.

## 1 · Configuration

`SEED` is fixed so the train/eval split is reproducible from the cache
directory alone — the cache is content-addressed by prompt hash, so the same
`data/llm_cache/` always yields the same split.

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"

SEED         = 20260829
EVAL_FRAC    = 0.15          # ~20 of 134 held out

# LoRA. r=16 is the smallest rank that reliably learns a JSON schema this
# nested; r=8 dropped fields in a way that looked like model weakness but was
# capacity. alpha = 2r is the usual scaling.
LORA_R       = 16
LORA_ALPHA   = 32

# Training. 134 examples is a SMALL set: the goal is format and field
# discipline, not domain knowledge. 3 epochs, low LR, and stop.
EPOCHS       = 3
LR           = 2e-4
BATCH        = 1             # 16 GB, 4096-token sequences
GRAD_ACCUM   = 8             # effective batch 8
MAX_SEQ_LEN  = 4096          # prompt (~3.5k tok) + response (~600 tok)

ADAPTER_DIR  = "/kaggle/working/extractor-qwen3b"

# Point this at the repo's cache. On Kaggle, upload data/llm_cache as a dataset.
CACHE_DIR    = "/kaggle/input/researchgpt-llm-cache"
PROMPT_PATH  = "/kaggle/input/researchgpt-prompts/extractor_v1.md"

## 2 · Dependencies

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Install only what is absent. A re-run costs seconds, not minutes."""
    try:
        importlib.import_module(import_name or pkg.split("==")[0].split(">")[0].replace("-", "_"))
        print(f"  present : {pkg}")
    except ImportError:
        print(f"  install : {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# transformers/torch/datasets ship with the Kaggle image; the PEFT stack does not.
for pkg, mod in [
    ("transformers>=4.44", "transformers"),
    ("datasets", "datasets"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("bitsandbytes", "bitsandbytes"),
    ("trl", "trl"),
]:
    ensure(pkg, mod)

import torch
print()
print("torch      :", torch.__version__)
print("cuda       :", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("device     :", name, f"({vram:.1f} GB)")
    # T4 is sm_75: NO bf16. Every cell below uses fp16 for this reason.
    print("bf16 ok    :", torch.cuda.is_bf16_supported(), "  <- False on T4, as expected")

## 3 · Build the distillation set from the LLM cache

Each cache entry is `{model, provider, prompt, response, ts}`. The prompt is the
**fully rendered `extractor_v1` template** — schema, six few-shot examples and
the paper text — and the response is the teacher's JSON. That pair is the
training example: no re-rendering, no reconstruction, so the student learns
against exactly the prompt `agents/extract.py` will send it.

Three filters, in order:

1. **Extractor calls only.** The cache also holds planner calls.
2. **Parseable teacher output.** A response the project's own `extract_json`
   cannot recover is not a label — it is a teacher failure, and training on it
   teaches the failure.
3. **No placeholder echo.** A teacher response containing the literal string
   `"string"` in a content field is fabricated content (T-004 disqualified an
   arm for exactly this). Training on it would teach the student to fabricate
   in a pipeline graded on not fabricating. **This filter is not optional.**

In [ ]:
import json, pathlib, random, re, hashlib
from collections import Counter

PLACEHOLDERS = {"string", "string | null", "null", "none", "n/a", "na", "...",
                "example", "text", "value", "<string>"}
CONTENT_FIELDS = ("methodology", "advantages", "disadvantages", "critique",
                  "datasets", "metrics", "results", "limitations")

_FENCE_RE = re.compile(r"```(?:json)?\s*(\{.*?\})\s*```", re.DOTALL | re.I)
_BARE_RE  = re.compile(r"\{.*\}", re.DOTALL)

def extract_json(text):
    """Same recovery ladder as researchgpt.core.llm.extract_json."""
    raw = (text or "").strip()
    if not raw:
        raise ValueError("empty")
    m = _FENCE_RE.search(raw)
    for cand in ([m.group(1)] if m else []) + [raw]:
        try:
            obj = json.loads(cand.strip())
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            pass
    m2 = _BARE_RE.search(raw)
    if m2:
        obj = json.loads(m2.group(0))
        if isinstance(obj, dict):
            return obj
    raise ValueError("no JSON object")

def has_echo(obj):
    for f in CONTENT_FIELDS:
        v = obj.get(f)
        if isinstance(v, str) and v.strip().lower() in PLACEHOLDERS:
            return True
        if isinstance(v, list) and any(
                isinstance(x, str) and x.strip().lower() in PLACEHOLDERS for x in v):
            return True
    return False

rows, reasons = [], Counter()
for p in sorted(pathlib.Path(CACHE_DIR).glob("*.json")):
    try:
        d = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        reasons["unreadable"] += 1
        continue
    prompt, response = d.get("prompt") or "", d.get("response") or ""

    if '"methodology"' not in prompt:          # planner / writer call
        reasons["not-extractor"] += 1
        continue
    try:
        obj = extract_json(response)
    except ValueError:
        reasons["teacher-unparseable"] += 1
        continue
    if not any(obj.get(f) for f in CONTENT_FIELDS):
        reasons["teacher-empty"] += 1
        continue
    if has_echo(obj):
        reasons["placeholder-echo"] += 1       # DROPPED - see the markdown above
        continue

    rows.append({
        "prompt": prompt,
        # Re-serialise from the parsed object rather than keeping the raw
        # string: this strips the teacher's markdown fences and trailing prose,
        # so the student is taught to emit bare JSON and nothing else.
        "completion": json.dumps(obj, ensure_ascii=False, indent=2),
        "teacher": d.get("model"),
        "n_fields": sum(1 for f in CONTENT_FIELDS if obj.get(f)),
    })

print("kept:", len(rows))
print("dropped:", dict(reasons))
print("teacher mix:", dict(Counter(r["teacher"] for r in rows)))
print("mean fields of 8:", round(sum(r["n_fields"] for r in rows) / max(1, len(rows)), 2))

### Split

Split on a **hash of the prompt**, not `random.shuffle`. The cache is
content-addressed, so a hash split is stable when new cache entries arrive —
re-running after another pipeline run adds examples to train without silently
moving old ones across the boundary and contaminating the eval set.

In [ ]:
def bucket(row):
    h = hashlib.sha256(row["prompt"].encode("utf-8")).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF

train_rows = [r for r in rows if bucket(r) >= EVAL_FRAC]
eval_rows  = [r for r in rows if bucket(r) <  EVAL_FRAC]
print(f"train {len(train_rows)}  eval {len(eval_rows)}")

if len(train_rows) < 60:
    print("\n  WARNING: under 60 training examples. LoRA will overfit the")
    print("  few-shot examples rather than the task. Run the pipeline a few")
    print("  more times to grow data/llm_cache before trusting this.")

## 4 · Format as chat

Qwen2.5-Instruct expects its own chat template. The cached prompt goes in as
the **user turn verbatim** — it already contains the system-level instructions,
the schema and the few-shot examples, because that is what the pipeline sends.

**Loss is masked to the completion only.** Training on the prompt tokens would
spend most of the gradient teaching the model to reproduce its own few-shot
examples, which is 14,125 characters of text it will always be given anyway.

In [ ]:
from datasets import Dataset

def to_chat(row):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]},
         {"role": "assistant", "content": row["completion"]}],
        tokenize=False, add_generation_prompt=False,
    )
    # The prefix up to the assistant turn is what gets masked out.
    prefix = tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    return {"text": text, "prompt_only": prefix}

train_ds = Dataset.from_list(train_rows).map(to_chat)
eval_ds  = Dataset.from_list(eval_rows).map(to_chat)

lens = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"][:20]]
print("token length p50/p90/max over 20 samples:",
      sorted(lens)[len(lens)//2], sorted(lens)[int(len(lens)*0.9)], max(lens))
print(f"MAX_SEQ_LEN = {MAX_SEQ_LEN} - raise it if max exceeds this, or examples truncate")

## 5 · Load Qwen in 4-bit and attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    # fp16, not bf16 - the T4 is sm_75 and has no bf16 path. Using bf16 here
    # silently falls back to fp32 accumulation and doubles the step time.
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Causal LM + right padding corrupts the loss mask on the last token. Qwen's
# own chat template assumes left padding for generation.
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False           # incompatible with gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # All seven projections. Attention-only (q,v) is the common shortcut and it
    # underfits format-learning tasks like this one, where the MLP carries most
    # of the "what shape is the output" signal.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6 · Train

`paged_adamw_8bit` and gradient checkpointing are what keep a 3B QLoRA run
inside a T4's 16 GB at 4096-token sequences. Neither is optional here.

Expect **35–60 minutes** for 3 epochs over ~114 examples. Kaggle's free GPU
quota is 30 h/week, so this is cheap; the constraint is the 12 h session limit,
which this is nowhere near.

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,               # T4: fp16, never bf16
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,           # packing would blend two papers into one example
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(ADAPTER_DIR)
print("adapter ->", ADAPTER_DIR)

## 7 · Evaluate on the project's own metrics

Loss is not the acceptance criterion. **T-004's four criteria are**, in T-004's
order of precedence:

| # | criterion | why it ranks here |
|---|---|---|
| 1 | placeholder echo | **disqualifying.** It parses, it validates, and it injects fabricated content |
| 2 | clean parse rate | JSON recovered with no `format: json` crutch |
| 3 | mean fields of 8 | the coverage number the hosted arm scores 4.6–4.9 on |
| 4 | latency | secondary; local latency is a deployment fact, not a quality one |

The comparison target is the **teacher on the same held-out papers**, not an
absolute. A student that matches the teacher has succeeded; one that exceeds it
is measuring noise on ~20 examples.

In [ ]:
import torch, time
from statistics import mean, median

model.config.use_cache = True
model.eval()

def generate(prompt, max_new_tokens=768):
    msgs = [{"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                        return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=None, top_p=None,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True), time.time() - t0

student, teacher, lat = [], [], []
for row in eval_rows:
    text, secs = generate(row["prompt"])
    lat.append(secs)
    try:
        obj = extract_json(text)
    except ValueError:
        student.append(None); teacher.append(extract_json(row["completion"])); continue
    student.append(obj)
    teacher.append(extract_json(row["completion"]))

parsed   = [o for o in student if o is not None]
echoes   = sum(1 for o in parsed if has_echo(o))
s_fields = [sum(1 for f in CONTENT_FIELDS if o.get(f)) for o in parsed]
t_fields = [sum(1 for f in CONTENT_FIELDS if o.get(f)) for o in teacher]

print(f"  held out            {len(eval_rows)} papers")
print(f"  clean parse rate    {len(parsed)/len(student):.0%}  ({len(parsed)}/{len(student)})")
print(f"  placeholder echoes  {echoes}" + ("   <- DISQUALIFYING" if echoes else "   <- clean"))
print(f"  student fields/8    {mean(s_fields):.2f}" if s_fields else "  student fields/8    n/a")
print(f"  TEACHER fields/8    {mean(t_fields):.2f}   <- the ceiling, not a target to beat")
print(f"  median latency      {median(lat):.1f}s on a T4 (the 5 GB card will be slower)")

## 8 · Adoption gate

Same shape as the reranker's gate in `03_train_reranker_mixed_negatives.ipynb`:
a written rule, decided before the numbers are seen.

**Adopt the student as the local extraction arm only if all four hold:**

1. `placeholder echoes == 0` — no exceptions, no "only one paper"
2. `clean parse rate >= 0.90`
3. `mean fields >= teacher mean - 0.5` — matching, not beating
4. it actually loads in Ollama at `q4_K_M` and clears `python -m agents.probe`

**If it fails, the hosted arm stands and that is a publishable result**: it
would say a 3B student cannot carry structured scientific extraction even with
the teacher's own outputs as labels, which is direct evidence for the project's
"architecture beats model scale" thesis rather than against it.

In [ ]:
gate = {
    "no_placeholder_echo": echoes == 0,
    "parse_rate_>=_0.90":  (len(parsed) / max(1, len(student))) >= 0.90,
    "fields_within_0.5":   bool(s_fields) and mean(s_fields) >= mean(t_fields) - 0.5,
}
for k, v in gate.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print()
print("VERDICT:", "ADOPT (pending the Ollama load test)" if all(gate.values()) else "REJECT - hosted arm stands")

## 9 · Export for Ollama

Training happened in 4-bit on a T4. **Inference happens on a 5 GB card through
Ollama**, which wants GGUF. Three steps: merge the adapter into fp16 weights,
convert to GGUF, quantise to `q4_K_M`.

`q4_K_M` is the target because it is what the VRAM table in `agents/README.md`
is sized against - a 3B at `q4_K_M` is ~1.9 GB of weights, leaving room for an
8192-token KV cache on a 5 GB card. `q8_0` would be better quality and would not
fit alongside the context window.

⚠️ **Merging must happen in fp16, not 4-bit.** `merge_and_unload()` on a 4-bit
model silently produces garbage weights - the dequantise-merge-requantise round
trip is not lossless and the adapter deltas are small enough to be destroyed by
it. Reload the base in fp16 first, which is why this cell is slow.

In [ ]:
import torch, gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Free the 4-bit training copy before loading the fp16 one - a T4 will not hold
# both.
try:
    del trainer, model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"": "cpu"},
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_fp16, "/kaggle/working/extractor-qwen3b").merge_and_unload()
merged.save_pretrained("/kaggle/working/extractor-qwen3b-merged", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/extractor-qwen3b-merged")
print("merged fp16 weights ->", "/kaggle/working/extractor-qwen3b-merged")

In [ ]:
# llama.cpp's converter. Kaggle has internet on, so clone it here rather than
# adding a dataset dependency.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp 2>/dev/null || true
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/extractor-qwen3b-merged \
    --outfile /kaggle/working/researchgpt-extractor-3b-f16.gguf \
    --outtype f16

# Quantise. The binary needs building once; ~3 min on Kaggle's CPUs.
!cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
!cmake --build /kaggle/working/llama.cpp/build --target llama-quantize -j4 > /dev/null

!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    /kaggle/working/researchgpt-extractor-3b-f16.gguf \
    /kaggle/working/researchgpt-extractor-3b-q4_k_m.gguf \
    Q4_K_M

!ls -lh /kaggle/working/*.gguf

### Loading it on the 5 GB machine

Download `researchgpt-extractor-3b-q4_k_m.gguf` from the Kaggle output, put it next to a
`Modelfile`, and register it with Ollama:

```
FROM ./researchgpt-extractor-3b-q4_k_m.gguf

# 8192, not Ollama's default 2048. See agents/README.md - at the default,
# Ollama silently drops the front of the prompt and answers from the remainder.
PARAMETER num_ctx 8192
PARAMETER temperature 0
# No SYSTEM line: extractor_v1.md carries its own instructions.
```

```bash
ollama create researchgpt-extractor:3b -f Modelfile
python -m agents.probe --model researchgpt-extractor:3b
```

Then the scripts in `agents/` can score it against the hosted arm with no
further change.